# Comparador Automatizado de Normativas vs Manuales Internos

**Pipeline de 5 fases** para analizar el cumplimiento de manuales internos bancarios
respecto a normativas ecuatorianas (SBS, BCE, SEPS, UAF).

| Fase | Descripción | Módulo |
|------|-------------|--------|
| 1 | Tabulación de documentos (PDF → DataFrame) | `document_parser` |
| 2 | Motor de búsqueda (FAISS semántico + léxico + reranker) | `search_engine` |
| 3 | Retrieve-then-Grade (validación de candidatos via LLM) | `llm_grader` |
| 4 | Análisis comparativo + NER (LLM) | `llm_grader` |
| 5 | Procesamiento concurrente (ThreadPoolExecutor + tqdm) | `comparator` |

**Modelos Docker Model Runner disponibles:**
- 🔵 Embedding: `ai/qwen3-embedding:latest` (2560 dim) — mejor calidad semántica en español
- 🔵 Embedding rápido: `ai/granite-embedding-multilingual:latest` (768 dim)
- 🟢 LLM: `docker.io/ai/gemma4:latest` — razonamiento interno (CoT), ideal para análisis legal
- 🟡 Reranker: `docker.io/ai/qwen3-reranker-vllm:0.6B` — filtrado post-FAISS

## ⚙️ 0. Configuración

In [ ]:
import os
import sys
import logging
from pathlib import Path

# Apple Silicon 16GB: evita el crash nativo del kernel cuando faiss y el
# runtime OpenMP de Docling/torch coexisten en el mismo proceso (Fase 2.2)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# Agregar src/ al path
sys.path.insert(0, str(Path.cwd()))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)

# Directorios de entrada y salida
NORMATIVA_DIR = Path("Normativa2026")          # PDFs de normativas
MANUAL_DIR    = Path("document_test")           # PDFs de manuales internos
OUTPUT_DIR    = Path("output/comparador")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Normativas: {list(NORMATIVA_DIR.glob('*.pdf'))}")
print(f"Manuales:   {list(MANUAL_DIR.glob('*.pdf'))}")

In [ ]:
# Importar todos los módulos del comparador
from src import (
    NormativaParser,
    ManualParser,
    LangChainDMREmbeddings,
    SentenceTransformersEmbeddings,
    NormativaIndex,
    LLMGrader,
    DocumentComparator,
)
from src.config import (
    DMR_BASE_URL,
    DMR_EMBED_MODEL,
    DMR_LLM_MODEL,
    FAISS_TOP_K,
    RERANKER_TOP_N,
)

print("✅ Módulos cargados correctamente")
print(f"   Embedding model : {DMR_EMBED_MODEL}")
print(f"   LLM model       : {DMR_LLM_MODEL}")
print(f"   DMR base URL    : {DMR_BASE_URL}")

## 📄 Fase 1: Tabulación de Documentos

### 1.1 Normativas (Docling + regex)

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd

# cache_dir apunta a los markdowns ya generados por Docling (no requiere reconversión)
normativa_parser = NormativaParser(cache_dir="output/docling")

# Stems explícitos para evitar intentar convertir L1-XVI-cap-*.pdf (no cacheados → segfault MPS)
NORMATIVA_STEMS = [
    "PDL-DERECHOS-DIGITALES",
    "LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGURIDAD_202652616421988",
    "Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo",
    "Proyecto-de-Ley-Transformacion-Digital-y-Audiovisual",
    "Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870",
]
normativa_pdfs = [NORMATIVA_DIR / f"{s}.pdf" for s in NORMATIVA_STEMS]
normativa_frames = []

for pdf in tqdm(normativa_pdfs, desc="Parseando normativas"):
    df = normativa_parser.parse_pdf(pdf)
    normativa_frames.append(df)
    print(f"  {pdf.name}: {len(df)} elementos")

normativa_df = pd.concat(normativa_frames, ignore_index=True)
print(f"\nTotal: {len(normativa_df)} elementos normativos")
normativa_df.head(3)

In [ ]:
normativa_df.to_excel("output/docling/normativa_df.xlsx", index=False)

In [ ]:
# Resumen de la normativa
print("Distribución por tipo de elemento:")
display(normativa_df["tipo_elemento"].value_counts().to_frame("cantidad"))

print("\nDistribución por documento:")
display(normativa_df["doc_id"].value_counts().to_frame("elementos"))

# Guardar normativa procesada
normativa_df.to_json(
    OUTPUT_DIR / "normativa_tabulada.json",
    orient="records",
    force_ascii=False,
    indent=2,
)
print("\n✅ Normativa guardada en output/comparador/normativa_tabulada.json")

In [ ]:
normativa_df.query('doc_id=="PDL-DERECHOS-DIGITALES.pdf"').sample(1).values

### 1.2 Manuales internos (Docling + HybridChunker)

In [ ]:
manual_parser = ManualParser(device="cpu")  # MPS inestable en conversiones sin caché → CPU estable

manual_pdfs = sorted(MANUAL_DIR.glob("*.pdf"))
manual_frames = []

for pdf in tqdm(manual_pdfs, desc="Parseando manuales"):
    df = manual_parser.parse_pdf(pdf)
    manual_frames.append(df)
    print(f"  {pdf.name}: {len(df)} chunks")

manual_df = pd.concat(manual_frames, ignore_index=True)
print(f"\nTotal: {len(manual_df)} secciones del manual")
manual_df.head(3)

In [ ]:
# Vista previa del manual
print(f"Secciones totales: {len(manual_df)}")
print(f"Documentos: {manual_df['doc_id'].unique()}")
print("\nPrimeros chunks:")
display(manual_df[["chunk_id", "jerarquia", "titulo_seccion", "texto"]].head(5))

manual_df.to_json(
    OUTPUT_DIR / "manual_tabulado.json",
    orient="records",
    force_ascii=False,
    indent=2,
)
print("✅ Manual guardado en output/comparador/manual_tabulado.json")

## 🔍 Fase 2: Motor de Búsqueda

### 2.1 Inicializar backend de embeddings

In [ ]:
# ai/qwen3-embedding (2560 dim) — preferido, pero requiere GPU funcional
# ai/granite-embedding-multilingual (768 dim) — alternativa estable
embedding_backend = LangChainDMREmbeddings(
    model="ai/granite-embedding-multilingual:latest",   # 768 dim, estable
    # model="ai/qwen3-embedding:latest",               # 2560 dim, mayor calidad
    base_url=DMR_BASE_URL,
)

import numpy as np
test_vec = embedding_backend.encode(["prueba de conexión DMR"])
print(f"✅ Embedding backend OK")
print(f"   Modelo    : {embedding_backend.model_name}")
print(f"   Dimensión : {test_vec.shape[1]}")
print(f"   Norma L2 (debe ≈ 1.0): {np.linalg.norm(test_vec[0]):.4f}")

### 2.2 Construir índice FAISS sobre la normativa

In [ ]:
from tqdm.notebook import tqdm as tqdmn

normativa_index = NormativaIndex(
    embedding_backend=embedding_backend,
    use_reranker=True,  # CrossEncoder local (sentence-transformers) — vllm-metal no soporta reranking
)

# Solo indexar artículos (excluir disposiciones/anexos si se prefiere)
# normativa_arts = normativa_df[normativa_df["tipo_elemento"] == "articulo"]

print("Construyendo índice FAISS...")
normativa_index.build(normativa_df, text_col="embed_text")

# Persistir en disco
normativa_index.save(OUTPUT_DIR / "faiss_index")
print(f"✅ Índice FAISS guardado en output/comparador/faiss_index/")

### 2.3 Verificar búsquedas

In [ ]:
# Test: búsqueda semántica
query = "política de crédito y evaluación de riesgo crediticio"
resultados = normativa_index.semantic_search(query, top_k=5)

print(f"Búsqueda: '{query}'")
print(f"Top-{len(resultados)} resultados FAISS:")
for r in resultados:
    print(f"  [{r['rank_faiss']}] Art.{r.get('numero','?')} (sim={r['similarity']:.4f}): {r.get('encabezado','')[:60]}")

In [ ]:
# Test: reranking post-FAISS
reranked = normativa_index.rerank(query, resultados, top_n=3)

print(f"Top-{len(reranked)} tras reranking:")
for r in reranked:
    score_str = f"reranker={r.get('reranker_score', 'N/A')}" if 'reranker_score' in r else f"sim={r.get('similarity', 0):.4f}"
    print(f"  [{r.get('rank', '?')}] Art.{r.get('numero','?')} ({score_str}): {r.get('encabezado','')[:60]}")

In [ ]:
# Test: búsqueda léxica
texto_manual = "Según el artículo 5 y el artículo 12 de la normativa vigente, el banco debe..."
lexical = normativa_index.lexical_scan(texto_manual, normativa_df)

print(f"Referencias léxicas detectadas: {len(lexical)}")
for m in lexical:
    print(f"  Art.{m.get('numero','?')}: {m.get('encabezado','')[:70]}")

## 🤖 Fase 3+4: LLM Grader — Retrieve-then-Grade + Análisis Comparativo

In [ ]:
llm_grader = LLMGrader(
    model=DMR_LLM_MODEL,   # docker.io/ai/gemma4:latest
    base_url=DMR_BASE_URL,
    temperature=0.0,
)

print(f"✅ LLM Grader inicializado")
print(f"   Modelo: {DMR_LLM_MODEL}")
print(f"   Nota: gemma4 tiene razonamiento interno (CoT) → latencia mayor, mayor calidad")

In [ ]:
# Test grading en una sección del manual
sample_row = manual_df.iloc[0].to_dict()
sample_text = sample_row.get("texto", "")

print(f"Sección del manual: {sample_row.get('jerarquia','')[:80]}")
print(f"Texto (primeros 300 chars): {sample_text[:300]}\n")

# Búsqueda de candidatos
candidates = normativa_index.semantic_search(sample_text, top_k=5)
candidates = normativa_index.rerank(sample_text, candidates, top_n=3)

print(f"Candidatos FAISS recuperados: {len(candidates)}")
for c in candidates:
    print(f"  Art.{c.get('numero','?')}: {c.get('encabezado','')[:60]}")

In [ ]:
# Grading de candidatos
print("Evaluando relevancia de candidatos...")
graded = llm_grader.grade_candidates(sample_text, candidates)

print("\nResultados del grading:")
for g in graded:
    status = "✅" if g.get("relevante") else "❌"
    print(f"  {status} Art.{g.get('numero','?')} | score={g.get('score_grade',0):.2f} | {g.get('razon_grade','')[:70]}")

In [ ]:
# Análisis comparativo completo
lexical_matches = normativa_index.lexical_scan(sample_text, normativa_df)
validated = [c for c in graded if c.get("relevante", True)]

print("Ejecutando análisis comparativo...")
analysis = llm_grader.analyze_comparison(sample_row, lexical_matches, validated)

print(f"\n{'='*60}")
print(f"Tipo de coincidencia : {analysis.tipo_coincidencia}")
print(f"Nivel de cumplimiento: {analysis.nivel_cumplimiento}")
print(f"\nAnálisis general:")
print(analysis.analisis_general)
if analysis.brechas:
    print(f"\nBrechas detectadas ({len(analysis.brechas)}):")
    for b in analysis.brechas:
        print(f"  • {b}")
if analysis.entidades_normativas:
    print(f"\nEntidades normativas: {analysis.entidades_normativas}")
if analysis.entidades_financieras:
    print(f"Entidades financieras: {analysis.entidades_financieras}")

## ⚡ Fase 5: Pipeline Completo con Procesamiento Concurrente

In [ ]:
comparator = DocumentComparator(
    normativa_index=normativa_index,
    llm_grader=llm_grader,
    top_k_faiss=FAISS_TOP_K,
    top_n_rerank=RERANKER_TOP_N,
)

print("DocumentComparator inicializado")
print(f"  FAISS top-k     : {FAISS_TOP_K}")
print(f"  Reranker top-n  : {RERANKER_TOP_N}")

In [ ]:
# Muestra rápida de 3 secciones para validar el pipeline antes del run completo
print("Ejecutando pipeline en muestra de 3 secciones...")
sample_results = comparator.run_sample(
    manual_df=manual_df,
    normativa_df=normativa_df,
    n=3,
    max_workers=1,
)

print(f"\n✅ Muestra completada: {len(sample_results)} secciones analizadas")
display(sample_results[["chunk_id", "jerarquia", "tipo_coincidencia", "nivel_cumplimiento", "analisis_general"]].head())

In [ ]:
# Pipeline sobre muestra de validación
# M1 16GB: gemma4 CoT secuencial en GPU → ~2-4 min/chunk (grading + análisis)
# n=5 → ~20-40 min total; aumentar a n=20 solo si se confirma estabilidad
print("Ejecutando pipeline sobre muestra de validación (n=5, max_workers=1)...")
results_df = comparator.run_sample(
    manual_df=manual_df,
    normativa_df=normativa_df,
    n=5,
    max_workers=1,
)

print(f"\n✅ Pipeline completado: {len(results_df)} secciones analizadas")
display(results_df[["chunk_id", "jerarquia", "tipo_coincidencia", "nivel_cumplimiento"]].head(5))

## 📊 Resultados y Exportación

In [ ]:
# Resumen estadístico
print("=== RESUMEN DE CUMPLIMIENTO ===")
summary = DocumentComparator.summary(results_df)
display(summary)

print("\n=== DISTRIBUCIÓN POR TIPO DE COINCIDENCIA ===")
display(results_df["tipo_coincidencia"].value_counts().to_frame("secciones"))

In [ ]:
# Secciones con omisiones críticas
omisiones = results_df[
    results_df["nivel_cumplimiento"].isin(["omision", "parcial"])
].copy()

print(f"Secciones con omisiones/cumplimiento parcial: {len(omisiones)}")
display(omisiones[["jerarquia", "tipo_coincidencia", "nivel_cumplimiento", "analisis_general"]].head(10))

In [ ]:
# Exportar a Excel formateado
excel_path = comparator.export_excel(
    results_df,
    output_path=OUTPUT_DIR / "reporte_comparacion.xlsx",
)
print(f"✅ Reporte exportado: {excel_path}")

# También guardar JSON completo
results_df.to_json(
    OUTPUT_DIR / "reporte_comparacion.json",
    orient="records",
    force_ascii=False,
    indent=2,
)
print(f"✅ JSON exportado: {OUTPUT_DIR / 'reporte_comparacion.json'}")

## 🔄 Uso Modular (sin re-ejecutar todo el pipeline)

Para análisis incrementales, carga el índice FAISS desde disco:

In [ ]:
# Cargar índice existente sin re-indexar
# IMPORTANTE: usar granite (768d) para coincidir con el índice FAISS guardado
from src import LangChainDMREmbeddings, NormativaIndex, LLMGrader, DocumentComparator
from src.config import DMR_BASE_URL, DMR_LLM_MODEL

embed_backend = LangChainDMREmbeddings(
    model="ai/granite-embedding-multilingual:latest",   # 768d — coincide con índice guardado
    base_url=DMR_BASE_URL,
)

saved_index = NormativaIndex(embed_backend)
saved_index.load(OUTPUT_DIR / "faiss_index")

grader = LLMGrader(model=DMR_LLM_MODEL, base_url=DMR_BASE_URL)
comp = DocumentComparator(saved_index, grader)

print("✅ Pipeline cargado desde disco")
print(f"   Vectores en índice: {saved_index._index.ntotal}")
print(f"   Dimensión embedding: {saved_index._index.d}")

In [ ]:
# Análisis incremental: validar recarga sin re-indexar
# Usar run_sample(n=3) para validación rápida (max_workers=1 para no sobrecargar DMR)
import pandas as pd

normativa_df = pd.read_json(OUTPUT_DIR / "normativa_tabulada.json", orient="records")
new_manual_df = pd.read_json(OUTPUT_DIR / "manual_tabulado.json", orient="records")

incremental = comp.run_sample(
    manual_df=new_manual_df,
    normativa_df=normativa_df,
    n=3,
    max_workers=1,
)

display(incremental[["chunk_id", "nivel_cumplimiento", "analisis_general"]].head())

## Comparativa pruebas

In [ ]:
incremental.head()